# 参数与扫描

这是独立的合成沙盒，不连接设备。无需完成其他课程。先从新 kernel 顺序运行，再做文末的小修改。
启动入口已准备环境与服务；重复打开继续当前练习，重置会创建新的起点。

In [ ]:
import scopecat as sc

session = sc.notebook()
session

In [ ]:
import numpy as np
from my_experiment.setup import open_parameters
from my_experiment.teaching import teaching_rabi

params = open_parameters(session)

参数来自独立的合成起点。修改工作区、构造扫描和预览本身都不采集。

In [ ]:
from my_experiment.parameters import Drive

params[Drive]["q0"].frequency = 5.145
saved = params.save(note="参数沙盒")
request = teaching_rabi().sweep(amplitude=np.linspace(0, 0.8, 7))
prepared = session.prepare(request, parameters=params)
print("预览点数:", prepared.preview.point_count)
assert prepared.preview.point_count == 7

运行后，每个扫描点保存 64 个 shot。下面同时显示复数均值，原始 shot 仍保留。

In [ ]:
run = prepared.run().wait(timeout=120).result()
shots = np.asarray(run.measurements()["iq"].require_values())
assert shots.shape == (7, 64)
print("run:", run.id)
print("每个扫描点的平均 IQ:", shots.mean(axis=1))

小修改：把扫描改为 5 点，重新预览和运行，观察形状。上面的 7 点断言也要随预期修改。
关闭 Notebook 前可运行任务“停止实验服务”；再次使用沙盒入口会重新启动。

参数声明在 `src/my_experiment/parameters.py`；实验在 `teaching.py`，初始化在 `setup.py`。它们都是本项目代码，可以查看和修改。

下面声明一张独立表，与 Drive 共存。同名表不能被当作另一张表；改变已有列的单位或主键需要显式结构操作。先保存已有值修改，再声明新表。

In [ ]:
class Readout(sc.ParameterModel, table="my_readout"):
    id: sc.Param[str] = sc.param(key=True)
    frequency: sc.Magnitude[float] = sc.quantity(unit="GHz")


params.declare_table(Readout)
if "q0" not in params[Readout]:
    params[Readout].add(Readout(id="q0", frequency=5.146))
params.save(note="自己的参数表")
assert params[Drive]["q0"].frequency == 5.145
assert params[Readout]["q0"].frequency == 5.146
params

小扩展：把 Readout 声明移到本地 `parameters.py`，将实验中的 `sc.parameter_ref(Drive.frequency, "q0")` 替换为 `sc.parameter_ref(Readout.frequency, "q0")`，并在实验模块中 import Readout。新增表不会自动接入实验，只有显式使用的参数才影响结果。

Notebook 默认工作区只为新请求选择已保存代码；旧请求和预览保留原版本。初始化只设置新表的起点，不会覆盖你已保存的参数。